# Predicting Denver Traffic Accident Severity

In [60]:
import pandas as pd
import numpy as np


from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

In [61]:
# Load data.
df = pd.read_csv("traffic_accident_data.csv", low_memory=False)

# Create the target variable before dropping leakage columns.
# Create high risk variable.

# Fill NaN as 0 so the comparison is safe
si = df["SERIOUSLY_INJURED"].fillna(0)
fat = df["FATALITIES"].fillna(0)

# Create a field where any fatality or serious injury
# is classified as "high risk".
df["high_risk"] = ((si > 0) | (fat > 0)).astype(int)

In [62]:
# Drop columns. I am taking a guess by removing these
# columns which appear superflous. The leakage columns
# indicate data we are trying to predict, so we omit them.

ID_COLS = [
    "object_id", "incident_id", "offense_id",
    "incident_address", "reported_date",
    "last_occurrence_date", "POINT_X", "POINT_Y", "x", "y",
    "offense_code", "offense_code_extension",
]

LEAKAGE_COLS = [
    "SERIOUSLY_INJURED", "FATALITIES",
    "FATALITY_MODE_1", "FATALITY_MODE_2",
    "SERIOUSLY_INJURED_MODE_1", "SERIOUSLY_INJURED_MODE_2",
    "HARMFUL_EVENT_SEQ_1", "HARMFUL_EVENT_SEQ_2", "HARMFUL_EVENT_SEQ_3",
]

cols_to_drop = [c for c in ID_COLS + LEAKAGE_COLS]
df.drop(columns=cols_to_drop, inplace=True)

In [ ]:
#print(df["first_occurrence_date"].unique())

In [47]:
# It is necessary to separate the time of the incident 
# into multiple categories.

df["first_occurrence_date"] = pd.to_datetime(
    df["first_occurrence_date"], errors="coerce"
)

df["hour"] = df["first_occurrence_date"].dt.hour
df["day_of_week"] = df["first_occurrence_date"].dt.dayofweek   # 0=Mon 6=Sun
df["month"] = df["first_occurrence_date"].dt.month

# We no longer need the raw datetime column
df.drop(columns=["first_occurrence_date"], inplace=True)

Extracted: hour, day_of_week, month, is_weekend


In [50]:
# Clean categorical columns

# selecting for obkect dtype gives us features with strings in them 
# These will be one hot encoded.
cat = df.select_dtypes(include="object").columns.tolist()

# Normalize strings
for col in cat:
    df[col] = (
        df[col]
        .astype(str)         # ensure string type
        .str.strip()         # remove leading/trailing whitespace
        .str.upper()         # normalize to uppercase
        .replace("NAN", np.nan)   # convert "nan" strings back to NaN
        .replace("", np.nan)      # blank strings -> NaN
    )

In [51]:
# Handle missing values

# Separate feature types (excluding the target)
feature_cols = [c for c in df.columns if c != "high_risk"]
cat_cols  = df[feature_cols].select_dtypes(include="object").columns.tolist()
num_cols  = df[feature_cols].select_dtypes(include=["number"]).columns.tolist()

# Fill categoricals with "UNKNOWN". By doing this, we can keep dataponits we would
# otherwise have to drop.
df[cat_cols] = df[cat_cols].fillna("UNKNOWN")

# Fill numeric columns with median. It's a safe choice.
for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)


In [52]:
# Separate features X and target y

X = df.drop(columns=["high_risk"])
y = df["high_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,      # keep class balance in both splits
)

# Re-identify column types from training data only
final_cat_cols = X_train.select_dtypes(include="object").columns.tolist()
final_num_cols = X_train.select_dtypes(include="number").columns.tolist()

In [56]:
# Build ColumnTransformer preprocessor

preprocessor = ColumnTransformer(
    transformers=[
        # One-hot encode categorical columns; ignore unseen categories
        ("cat", OneHotEncoder(handle_unknown="ignore"), final_cat_cols),
        # Scale numeric columns to zero mean / unit variance
        ("num", StandardScaler(), final_num_cols),
    ],
    remainder="drop",   # Discard any column not listed above.
)

In [66]:
# Define models

results = []

# Model 1, a dummy classifier that will always choose low risk.
# Acts as a control group.
dummy_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent", random_state=42)),
])

# Model 2, logistic regression
lr_pipe = Pipeline([
    ("preprocessor", preprocessor),
    # max_iter=1000 to ensure convergence; class_weight='balanced' helps
    # with imbalanced classes by penalising misclassifying the minority more
    ("classifier", LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42
    )),
])